In [1]:
from abc import ABC, abstractmethod
from datetime import datetime

In [2]:
class SmartDevice(ABC):
    _device_count = 0
    _devices = set()

    def __init__(self, device_id):
        if device_id in SmartDevice._devices:
            print(f"Device with id: {device_id} already exist")
            return
        self._device_id = device_id
        self.__is_on = False
        SmartDevice._device_count += 1
        SmartDevice._devices.add(device_id)

    def turn_on(self):
        if not self.__is_on:
            self.__is_on = True
            print(f"Device {self._device_id} turned on.")
        else:
            print(f"Device {self._device_id} is already on.")

    def turn_off(self):
        if self.__is_on:
            self.__is_on = False
            print(f"Device {self._device_id} turned off.")
        else:
            print(f"Device {self._device_id} is already off.")

    @property
    def is_on(self):
        return self.__is_on

    @classmethod
    def get_device_count(cls):
        return cls._device_count

    @staticmethod
    def get_system_time():
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    @abstractmethod
    def get_status_report(self):
        pass

    @abstractmethod
    def perform_action(self, action_type, value=None):
        pass


In [3]:
class SmartLight(SmartDevice):
    def __init__(self, device_id):
        super().__init__(device_id)
        self.__brightness = 0

    @property
    def brightness(self):
        return self.__brightness

    @brightness.setter
    def brightness(self, level):
        if not self.is_on:
            print("light is not on. Brightness level cannot be changed")
        elif 0 <= level <= 100:
            self.__brightness = level
            print(f"Brightness set to {level}.")
        else:
            print("Brightness level must be between 0 and 100.")

    def get_status_report(self):
        return f"[SmartLight] ID: {self._device_id}, ON: {self.is_on}, Brightness: {self.__brightness}"

    def perform_action(self, action_type, value=None):
        if action_type.lower() == "set_brightness":
            self.brightness = value
        else:
            print(f"Unknown action '{action_type}' for SmartLight.")


In [4]:
class SmartThermostat(SmartDevice):
    def __init__(self, device_id):
        super().__init__(device_id)
        self.__temperature = 20.0

    @property
    def temperature(self):
        return self.__temperature

    @temperature.setter
    def temperature(self, temp):
        if not self.is_on:
            print("Thermostat is not on. Temperature cannot be changed.")
        elif 18.0 <= temp <= 30.0:
            self.__temperature = temp
            print(f"Temperature set to {temp}C.")
        else:
            print("Temperature must be between 18.0C and 30.0C.")

    def get_status_report(self):
        return f"[SmartThermostat] ID: {self._device_id}, ON: {self.is_on}, Temperature: {self.__temperature}C"

    def perform_action(self, action_type, value=None):
        if action_type.lower() == "set_temperature":
            self.temperature = value
        else:
            print(f"Unknown action '{action_type}' for SmartThermostat.")


In [5]:
class Programmable(ABC):
    def schedule_task(self):
        pass

In [6]:
class SecuritySensor(SmartDevice):
    def __init__(self, device_id):
        super().__init__(device_id)
        self.__is_armed = False
    def is_armed(self):
        return self.__is_armed
        

    def arm_sensor(self):
        self.__is_armed = True
        print(f"Sensor {self._device_id} armed.")
    
    def disarm_sensor(self):
        self.__is_armed = False
        print(f"Sensor {self._device_id} disarmed.")
    def get_status_report(self):
        return f"[SecuritySensor] ID: {self._device_id}, ON: {self.is_on}, Armed: {self.__is_armed}"

    def turn_off(self):
        self.__is_armed = False
        super().turn_off()
    def perform_action(self, action_type, value=None):
        if action_type.lower() == "arm":
            self.__is_armed = True
            print(f"Sensor {self._device_id} armed.")
        elif action_type.lower() == "disarm":
            self.__is_armed = False
            print(f"Sensor {self._device_id} disarm.")
        else:
            print(f"Unknown action '{action_type}' for SecuritySensor.")


In [7]:
class SmartAlarmSystem(SmartDevice, Programmable):
    __sensors = list()
    device_id = ""
    def __init__(self, device_id):
        super().__init__(device_id)
        self.device_id = device_id
        self.__sensors.append(SecuritySensor(device_id = device_id+"."+str(len(self.__sensors)+1)))#has a relation(loose coupling)

    def add_sensor(self):
        self.__sensors.append(SecuritySensor(device_id = self.device_id+"."+str(len(self.__sensors)+1)))

    def remove_sensor(self,device_id = None):
        if(self.__sensors[0].is_armed()):
            print("Alarm System is armed cannot disconnect the sensors.")
            return
        if(device_id is None):
            sensor = self.__sensors.pop()
            print("A Sensor got Disconnected")
            return
    
    def get_status_report(self):
        sensor_status = ""
        for sensor in self.__sensors:
            sensor_status+=sensor.get_status_report()+"\n"
        return f"[SmartAlarmSystem] ID: {self._device_id}, ON: {self.is_on},\nArmed:"+f"\n{sensor_status}"

    def perform_action(self, action_type, value=None):
        if action_type == "arm":
            for sensor in self.__sensors:
                sensor.turn_on()
                sensor.perform_action(action_type,value)
            print(f"Alarm system {self._device_id} armed.")
        elif action_type == "disarm":
            for sensor in self.__sensors:
                sensor.turn_off()
                sensor.perform_action(action_type,value)
            print(f"Alarm system {self._device_id} disarmed.")
        else:
            print(f"Unknown action '{action_type}' for SmartAlarmSystem.")

    def schedule_task(self):
        print(f"Alarm system {self._device_id} task scheduled.")

    def turn_off(self):
        for sensor in self.__sensors:
            sensor.turn_off()
        super().turn_off()
            




In [8]:
class HomeManager:
    def __init__(self):
        self._devices = []

    def add_device(self, device):
        if(not isinstance(device,SmartDevice)):
            print("Device is not a SmartDevice.")
            return 
        self._devices.append(device)
        print(f"Device {device._device_id} added to HomeManager.")

    def control_device(self, device_id, action_type, value=None):
        for device in self._devices:
            if device._device_id == device_id:
                device.perform_action(action_type, value)
                return
        print(f"Device {device_id} not found.")

    def get_all_device_statuses(self):
        for device in self._devices:
            print(device.get_status_report())



In [9]:
manager = HomeManager()#home manager initialized 

In [10]:
light = SmartLight("SD001")
thermostat = SmartThermostat("SD002")
alarm = SmartAlarmSystem("SD003")

In [11]:
alarm.add_sensor()

In [12]:
manager.add_device(light)
manager.add_device(thermostat)
manager.add_device(alarm)

Device SD001 added to HomeManager.
Device SD002 added to HomeManager.
Device SD003 added to HomeManager.


In [13]:
light.turn_on()
thermostat.turn_on()
alarm.turn_on()

Device SD001 turned on.
Device SD002 turned on.
Device SD003 turned on.


In [14]:
manager.control_device("SD001", "set_brightness", 80)
manager.control_device("SD002", "set_temperature", 25.5)
manager.control_device("SD003", "arm")
manager.control_device("SD001", "set_brightness", 150)

Brightness set to 80.
Temperature set to 25.5C.
Device SD003.1 turned on.
Sensor SD003.1 armed.
Device SD003.2 turned on.
Sensor SD003.2 armed.
Alarm system SD003 armed.
Brightness level must be between 0 and 100.


In [15]:
print("\nDevice Status:")
manager.get_all_device_statuses()


Device Status:
[SmartLight] ID: SD001, ON: True, Brightness: 80
[SmartThermostat] ID: SD002, ON: True, Temperature: 25.5C
[SmartAlarmSystem] ID: SD003, ON: True,
Armed:
[SecuritySensor] ID: SD003.1, ON: True, Armed: True
[SecuritySensor] ID: SD003.2, ON: True, Armed: True



In [16]:
alarm.turn_off()

Device SD003.1 turned off.
Device SD003.2 turned off.
Device SD003 turned off.


In [17]:
light.turn_off()
device_count = SmartDevice.get_device_count()
print(f"\nTotal Smart Devices: {device_count}")

Device SD001 turned off.

Total Smart Devices: 5


In [18]:
print("\nDevice Status:")
manager.get_all_device_statuses()


Device Status:
[SmartLight] ID: SD001, ON: False, Brightness: 80
[SmartThermostat] ID: SD002, ON: True, Temperature: 25.5C
[SmartAlarmSystem] ID: SD003, ON: False,
Armed:
[SecuritySensor] ID: SD003.1, ON: False, Armed: False
[SecuritySensor] ID: SD003.2, ON: False, Armed: False



In [19]:
system_time = SmartDevice.get_system_time()
print(f"Current System Time: {system_time}")

Current System Time: 2025-06-30 18:44:04


In [20]:
print("\nMRO of SmartAlarmSystem:")
print(SmartAlarmSystem.__mro__)


MRO of SmartAlarmSystem:
(<class '__main__.SmartAlarmSystem'>, <class '__main__.SmartDevice'>, <class '__main__.Programmable'>, <class 'abc.ABC'>, <class 'object'>)
